## Open AI Agent SDK Working Example

In [1]:
from agents import (
    Agent,
    set_default_openai_api,
    function_tool,
    Runner,
    SQLiteSession,
)
import os
import nest_asyncio
from dotenv import load_dotenv
import requests

nest_asyncio.apply()
load_dotenv()
set_default_openai_api("chat_completions")  # use chat completions api
set_default_openai_api(os.getenv("OPENAI_API_KEY"))

### Example 1: Simple One Agent Call

In [2]:
history_tutor_agent = Agent(
    name="Historical Tutor",
    instructions="An agent that acts as a historical tutor, answering questions about historical events, figures, and contexts.",  # instruction is a system prompt
    model="gpt-4o-mini",
)

In [3]:
result = await Runner.run(
    history_tutor_agent,
    "Can you explain the causes and consequences of the French Revolution?",
)
print(result.final_output)

Certainly! The French Revolution, which began in 1789, was a pivotal event in world history that had complex causes and significant consequences.

### Causes of the French Revolution

1. **Social Inequality:** French society was divided into three estates:
   - **First Estate:** Clergy
   - **Second Estate:** Nobility
   - **Third Estate:** Commoners (about 97% of the population, including peasants, workers, and the bourgeoisie) faced heavy taxation and had little political power compared to the privileged classes.

2. **Economic Hardship:** France’s financial crisis was exacerbated by:
   - Costly wars, including participation in the American Revolutionary War.
   - Poor harvests in the late 1780s, leading to food shortages and rising bread prices.
   - A burden of debt, leading to demands for tax reforms.

3. **Enlightenment Ideas:** Philosophers like Rousseau, Voltaire, and Montesquieu promoted ideas about liberty, equality, and democracy, inspiring the Third Estate to seek a govern

### Example 2: Handoff

In [4]:
history_tutor_agent = Agent(
    name="Historical Tutor",
    handoff_description="Specialist agent for historical inquiries.",
    instructions="An agent that acts as a historical tutor, answering questions about historical events, figures, and contexts.",
    model="gpt-4o-mini",
)

math_tutor_agent = Agent(
    name="Math Tutor",
    handoff_description="Specialist agent for math inquiries.",
    instructions="An agent that acts as a math tutor, helping with mathematical problems and concepts.",
    model="gpt-4o-mini",
)

principal_agent = Agent(
    name="Principal Tutor",
    instructions="You determine which agent to use based on user's question",
    model="gpt-4o-mini",
    handoffs=[history_tutor_agent, math_tutor_agent],
)


In [5]:
result = await Runner.run(principal_agent, "what is the capital of China?")
print(result.final_output)

The capital of China is Beijing.


In [6]:
result = await Runner.run(principal_agent, "what is square of square root of 2?")
print(result.final_output)

To find the square of the square root of 2, you can follow these steps:

1. **Start with the square root of 2**: \(\sqrt{2}\)
  
2. **Now, square that result**: \((\sqrt{2})^2\)

According to the property of exponents, squaring a square root gives you the original number:

\[
(\sqrt{2})^2 = 2
\]

So, the square of the square root of 2 is **2**. If you have any more questions or need further explanation, feel free to ask!


In [7]:
for item in result.new_items:
    print(item)

HandoffCallItem(agent=Agent(name='Principal Tutor', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions="You determine which agent to use based on user's question", prompt=None, handoffs=[Agent(name='Historical Tutor', handoff_description='Specialist agent for historical inquiries.', tools=[], mcp_servers=[], mcp_config={}, instructions='An agent that acts as a historical tutor, answering questions about historical events, figures, and contexts.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks

### Example 3: Function Tools with Agents

In [8]:
@function_tool
async def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@function_tool
async def subtract(a: float, b: float) -> float:
    """subtract two numbers."""
    return a - b


@function_tool
async def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@function_tool
async def divide(a: float, b: float) -> float:
    """Divide two numbers."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b

In [9]:
calculator_agent = Agent(
    name="Calculator Agent",
    instructions="An agent that performs basic arithmetic operations using provided tools.",
    model="gpt-4o-mini",
    tools=[add, subtract, multiply, divide],
)

In [10]:
result = await Runner.run(calculator_agent, "What is (15 + 30) * 2 - 10 / 2 ?")
print(result.final_output)

The result of the expression \( (15 + 30) \times 2 - \frac{10}{2} \) is \( 90 - 8 = 82 \).


We can also define our own `run_agent` function as an async function.

Why using async method?
* Non-blocking I/O: Agents make multiple API calls (to OpenAI, tool executations, etc.) Async allows other code to run while waiting for responses.
* Better performance: When running multiple agents or queries, async enables concurrent executation instead of waiting for each sequentially
* OpenAI Agents SDK design: The SDK is built with async/await to handle the agent's iterative reasoning loop efficiently (think->call tool->think->respond)

The await flow:
1. Agent receives query
2. Agent thinks (OpenAI API call)
3. Agent calls tools 
4. Agent generates response
5. `await` completes, return full results 

In [11]:
async def run_agent(agent: Agent, query: str) -> str:
    return await Runner.run(agent, query)

In [12]:
result = await run_agent(calculator_agent, "What is 100 divided by 4 plus 6 times 3?")
print(result.final_output)
for item in result.new_items:
    print(item)

The result of \(100 \div 4 + 6 \times 3\) is \(43\).
ToolCallItem(agent=Agent(name='Calculator Agent', handoff_description=None, tools=[FunctionTool(name='add', description='Add two numbers.', params_json_schema={'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'add_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1116df740>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='subtract', description='subtract two numbers.', params_json_schema={'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'subtract_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10f31

In [13]:
# without await, it returns a coroutine rather than the actual result
result = run_agent(calculator_agent, "What is 100 divided by 4 plus 6 times 3?")
# print(result.final_output)
result

<coroutine object run_agent at 0x111f26ce0>

### Example 4: Use Agents as tools

Rather than use handoff functionality, we use agent as a tool in this example.

In [14]:
history_tutor_agent = Agent(
    name="Historical Tutor",
    instructions="An agent that acts as a historical tutor, answering questions about historical events, figures, and contexts.",
    model="gpt-4o-mini",
)

math_tutor_agent = Agent(
    name="Math Tutor",
    instructions="An agent that acts as a math tutor, helping with mathematical problems and concepts.",
    model="gpt-4o-mini",
)

In [15]:
tutor_agent = Agent(
    name="Tutor Agent",
    instructions="Call the relevant tools based on the user's question.",
    model="gpt-4o-mini",
    tools=[
        history_tutor_agent.as_tool(
            tool_name="tutor_history",
            tool_description="Specialist agent for historical inquiries.",
        ),
        math_tutor_agent.as_tool(
            tool_name="tutor_math",
            tool_description="Specialist agent for math inquiries.",
        ),
    ],
)

In [16]:
result = await run_agent(tutor_agent, "What is 100 divided by 4 plus 6 times 3?")
result.final_output

"The result of \\(100 \\div 4 + 6 \\times 3\\) is \\(43\\). \n\nHere's how it's calculated:\n\n1. **Division**: \\(100 \\div 4 = 25\\)\n2. **Multiplication**: \\(6 \\times 3 = 18\\)\n3. **Addition**: \\(25 + 18 = 43\\)\n\nSo, the final answer is \\(43\\)."

In [17]:
result.new_items

[ToolCallItem(agent=Agent(name='Tutor Agent', handoff_description=None, tools=[FunctionTool(name='tutor_history', description='Specialist agent for historical inquiries.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'tutor_history_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x111e0c4a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='tutor_math', description='Specialist agent for math inquiries.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'tutor_math_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x111e0cfe0>, strict_json_schema=True, is_enabled=True, tool_in

In [18]:
result = await run_agent(tutor_agent, "What is the tiananmen square massacre?")
result.final_output

'The Tiananmen Square Massacre refers to a violent crackdown on pro-democracy protesters in Beijing, China, in June 1989. The protests, which began in April of that year, were primarily led by students advocating for political freedoms, freedom of speech, and economic reforms. The movement drew participants from various segments of society, demanding an end to corruption and more democratic reforms.\n\nOn June 3-4, 1989, the Chinese government declared martial law and deployed the military to forcibly remove the protesters from Tiananmen Square. The situation escalated dramatically as troops used tanks and live ammunition against unarmed civilians. The exact number of casualties is uncertain, with estimates ranging from several hundred to possibly thousands of deaths.\n\nThe incident resulted in widespread international condemnation and remains a highly sensitive and censored topic in China. The Chinese government continues to control information regarding the event, monitoring and sup

In [19]:
result.new_items

[ToolCallItem(agent=Agent(name='Tutor Agent', handoff_description=None, tools=[FunctionTool(name='tutor_history', description='Specialist agent for historical inquiries.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'tutor_history_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x111e0c4a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='tutor_math', description='Specialist agent for math inquiries.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'tutor_math_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x111e0cfe0>, strict_json_schema=True, is_enabled=True, tool_in

### Example 5: Multi-Turn Conversation with OpenAI agents SDK

In [20]:
# first turn
query_1 = "What is 100 divided by 4 plus 6 times 3?"
result = await run_agent(calculator_agent, query_1)
print(result.final_output)

# save the conversation history
messages = result.to_input_list()

# second turn
query_2 = "now subtract 10?"

# append the conversation history
messages.append({"role": "user", "content": query_2})

result = await run_agent(calculator_agent, messages)
print(result.final_output)

The result of \( 100 \div 4 + 6 \times 3 \) is \( 43 \).
After subtracting 10, the result is \( 33 \).


### Example: Tool is an API call, which agent can invoke.

The key pattern is:
* Tool = thin wrapper around an API call
* The agent calls the tool → the tool calls another API → result flows back to the agent.

#### Step 1. Create the risk engine API hosted by losthost (FastAPI)
See code in `/Users/admin/Documents/AgenticAI_Learning/fastapi_learning/main`

In [21]:
# test api call
requests.get("http://localhost:8000/pd?customer_id=123").json()

{'customer_id': 123, 'pd': 0.023}

Trick:

* `customer_id` is passed as a query parameter in the URL. This means:
* The FastAPI endpoint `/pd` is designed to accept a parameter named `customer_id`.
* When you call the API with `?customer_id=123`, FastAPI extracts the value `123` and makes it available to your endpoint handler as a function argument.

In [22]:
# equivalent to above
response = requests.get(
    "http://127.0.0.1:8000/pd", params={"customer_id": 123}, timeout=5
)
response.json()

{'customer_id': 123, 'pd': 0.023}

#### Step 2: Wrap the API Call as a Tool
What the following does: 

* Makes a real HTTP call
* Returns clean, structured data
* Is now callable by the agent

The agent does not know this is an API call. It just sees a “function that returns PD”.

In [23]:
@function_tool
def get_probability_of_default(customer_id: int) -> float:
    """Get the probability of default for a customer given their ID."""
    response = requests.get(
        "http://127.0.0.1:8000/pd", params={"customer_id": customer_id}, timeout=5
    )
    data = response.json()
    return data["pd"]

#### Step 3: Create an Agent That Can Use This Tool

In [24]:
credit_agent = Agent(
    name="Credit Risk Agent",
    instructions="An agent that assesses credit risk using external API calls.",
    model="gpt-4o-mini",
    tools=[get_probability_of_default],  # add the function tool
)

In [25]:
result = await run_agent(credit_agent, "want to check the pd for customer id 456")
print(result.final_output)

The probability of default (PD) for customer ID 456 is 0.026, or 2.6%.


#### Step 4: Expose the Agent via FastAPI

In [ ]:
## syc
response = requests.post(
    "http://127.0.0.1:8000/credit_risk_assessment",
    json={"question": "want to check the pd for customer id 456"},
)
print(response.json())

{'answer': 'The probability of default (PD) for customer ID 456 is 0.026, or 2.6%.'}


You only get a coroutine if you call the async function directly in Python code without await.
But when accessed as an API endpoint, FastAPI always returns the actual result, not a coroutine, because it manages the event loop and awaits the function for you.

Summary:

* Calling the endpoint via HTTP: you get the result (not a coroutine).
* Calling the async function directly in Python without await: you get a coroutine (not recommended).
* FastAPI always returns the awaited result to the client.

In [44]:
response = requests.post(
    "http://127.0.0.1:8000/credit_risk_assessment_async",
    json={"question": "want to check the pd for customer id 456"},
)
print(response.json())

{'answer': 'It seems there was a timeout issue while trying to check the probability of default for customer ID 456. Would you like to try again, or is there anything else you need assistance with?'}


In [ ]:
# response = requests.post(
#     "http://127.0.0.1:8000/credit_risk_assessment_raw",
#     "want to check the pd for customer id 456",
# )
# print(response.json())

{'detail': [{'type': 'missing', 'loc': ['query', 'request'], 'msg': 'Field required', 'input': None}]}
